# 03 model and trust layer

Design decisions: flight-grouped splits (20 train / 20 calibration / 20 test per family, 5 repetitions); `rx_*` columns excluded from the detector; temperature scaling on calibration windows; class-conditional conformal sets at the flight level at alpha 0.10 and 0.20; leave-one-subtype-out; Whelan live logs as external case studies. Result tables are committed under `reports/<run>`.

In [ ]:
# ============================================================
# BOOTSTRAP  (top of every notebook in this project)
# ============================================================
import os, sys, subprocess, shutil
from pathlib import Path

PROJECT   = "UAV_GNSS"
REPO_NAME = "uav-gnss-triage"

DRIVE_MOUNT = Path("/content/drive")
DRIVE_ROOT  = DRIVE_MOUNT / "MyDrive" / f"{PROJECT}_Research"
REPO_DIR    = DRIVE_ROOT / REPO_NAME

if not (DRIVE_MOUNT / "MyDrive").exists():
    from google.colab import drive
    drive.mount(str(DRIVE_MOUNT))

for dotfile in (".gitconfig", ".git-credentials"):
    src = DRIVE_ROOT / dotfile
    if src.exists():
        shutil.copy(src, Path.home() / dotfile)
cred = Path.home() / ".git-credentials"
if cred.exists():
    os.chmod(cred, 0o600)
subprocess.run(["git", "config", "--global", "credential.helper", "store"], check=False)

if REPO_DIR.exists():
    os.chdir(REPO_DIR)
    if str(REPO_DIR / "src") not in sys.path:
        sys.path.insert(0, str(REPO_DIR / "src"))
    import paths as P
print("CWD:", os.getcwd(), "| credentials:", cred.exists())


In [ ]:
RUN = "sih_flights_v2"
subprocess.run(["pip", "install", "-q", "xgboost", "scikit-learn", "scipy", "pandas"], check=True)
out = P.REPORTS / RUN
proc = subprocess.Popen([sys.executable, str(P.SRC / "sih_model.py"), "--features", str(P.FEATURES / RUN), "--out", str(out)],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="")
print("exit code:", proc.wait())
